In [ ]:
try:
    from databricks.sdk import WorkspaceClient
except ImportError:
    !pip install databricks-sdk
    from databricks.sdk import WorkspaceClient

In [ ]:
# Importing Libraries
import re
import ast
import json
import requests
import hashlib
import pandas as pd
import numpy as np
from databricks.sdk import WorkspaceClient
from bs4 import BeautifulSoup
from google.colab import userdata
from concurrent.futures import ThreadPoolExecutor, as_completed

# Defining functions

In [ ]:
# ----- # ----- # ----- Get Jobs for JPMC ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def jobs_jpmc(limit=25, offset=0):
    response = ''
    try:
        url = 'https://jpmc.fa.oraclecloud.com/hcmRestApi/resources/latest/recruitingCEJobRequisitions'
        params = {
            'onlyData': 'true',
            'expand': 'requisitionList.workLocation,requisitionList.otherWorkLocations,requisitionList.secondaryLocations,flexFieldsFacet.values,requisitionList.requisitionFlexFields',
            'finder': f'findReqs;siteNumber=CX_1001,facetsList=LOCATIONS%3BWORK_LOCATIONS%3BWORKPLACE_TYPES%3BTITLES%3BCATEGORIES%3BORGANIZATIONS%3BPOSTING_DATES%3BFLEX_FIELDS,limit={limit},sortBy=POSTING_DATES_DESC,offset={offset}'
        }
        response = requests.get(url, params=params)
        response_json = json.loads(response.text)
        if 'items' in response_json:
            return True, response_json
        else:
            return False, {'Error': response.text}
    except Exception as e:
        print(str(e))
        print(f"Unable to connect to JPMC: {response.text}")
        return False, {'Error': response.text}


# ----- # ----- # ----- Get Job details for JPMC ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def jobs_details_jpmc(record=''):
    response = ''
    try:
        if record['index']%250 == 0:
          print(record['index'])
        url = f'https://jpmc.fa.oraclecloud.com/hcmRestApi/resources/latest/recruitingCEJobRequisitionDetails?expand=all&onlyData=true&finder=ById;Id=%22{record['Id']}%22,siteNumber=CX_1001'
        payload = {}
        headers = {}
        response = requests.request("GET", url, headers=headers, data=payload)
        response_json = json.loads(response.text)
        if 'items' in response_json:
            return response_json['items'][0]
        else:
            return {'Error': response.text}
    except Exception as e:
        print(str(e))
        print(record['Id'])
        # print(f"Unable to connect to JPMC: {response.text}")
        return {'Error': response.text}


# ----- # ----- # ----- Fetch Job List ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def fetch_jobs():
    limit = 25
    offset = 0
    job_df_list = []

    # Perform Initial API Call
    job_bool, job_json = jobs_jpmc(limit, offset)

    # Start the loop
    while len(job_json['items'][0]['requisitionList']) != 0:
        if offset%250 == 0:
          print(f'Limit: {limit}, Offset: {offset}, Count:{limit + offset}')
        job_bool, job_json = jobs_jpmc(limit, offset)
        job_df_list.extend(job_json['items'][0]['requisitionList'])
        offset += limit

    # Concatenate the dataset and clean
    jobs = pd.DataFrame(job_df_list)
    jobs.to_parquet('/content/drive/MyDrive/Colab Notebooks/JobAI/jobs/jpmc_jobs.parquet', index=False)
    # jobs = jobs[['Id', 'Title', 'PostedDate', 'JobFamily', 'JobFunction', 'ShortDescriptionStr', 'PrimaryLocation', 'secondaryLocations']]
    # jobs.reset_index(inplace=True, drop=False)
    print(f'Total Jobs: {len(jobs)}')
    return jobs


# ----- # ----- # ----- Fetch Job Details ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def fetch_job_details(job_df):
    results = []

    # Parallel processing
    with ThreadPoolExecutor(max_workers=50) as executor:  # adjust workers based on API rate limits
        future_to_record = {executor.submit(jobs_details_jpmc, row): row for _, row in job_df.iterrows()}
        for future in as_completed(future_to_record):
            results.append(future.result())
    results_df = pd.DataFrame(results)
    results_df.to_parquet('/content/drive/MyDrive/Colab Notebooks/JobAI/jobs/jpmc_jobs_details.parquet', index=False)
    # results_df = results_df[['Id', 'JobSchedule', 'ExternalDescriptionStr', 'CorporateDescriptionStr', 'OrganizationDescriptionStr', 'BusinessUnit', 'requisitionFlexFields']]
    return results_df

In [ ]:
# ----- # ----- # ----- Cleanup HTML ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
# def clean_html(text):
#     if pd.isna(text):
#         return ""
#     soup = BeautifulSoup(text, "html.parser")
#     return soup.get_text(separator="\n", strip=True)

def clean_html(text):
    if not text or not isinstance(text, str):
        return ''
    soup = BeautifulSoup(text, "html.parser")
    for tag in soup.find_all(['p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'div', 'li']):
        tag.append('\n')
    cleaned_text = soup.get_text(separator="\n", strip=True)
    cleaned_text = re.sub(r'\n\s*\n+', '\n\n', cleaned_text)
    return cleaned_text.strip()


# ----- # ----- # ----- Cleanup Data - Secondary Location ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def extract_locations(val):
    # if pd.isna(val):
    #     return ''

    if isinstance(val, list) and len(val) == 0:
      return ''

    # if val == [] or val == "[]" or (isinstance(val, (list, np.ndarray)) and len(val) == 0):
    #     return ''

    try:
        data = ast.literal_eval(val) if isinstance(val, str) else val
        if isinstance(data, list) and data:
            return "\n".join(d['Name'] for d in data if isinstance(d, dict) and 'Name' in d)
    except (ValueError, SyntaxError):
        return ''
    return ''


# ----- # ----- # ----- Cleanup Data - Salary ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def extract_salary(val):
    if not val or val == "[]" or val == "None":
        return ''
    try:
        data = ast.literal_eval(val) if isinstance(val, str) else val
        extracted_lines = []
        for entry in data:
            if entry.get('Prompt') == 'Base Pay/Salary':
                raw_value = entry.get('Value', '')
                parts = raw_value.replace(";", "\n").split("\n")
                extracted_lines.extend([p.strip() for p in parts if p.strip()])
        return "\n".join(extracted_lines) if extracted_lines else None
    except (ValueError, SyntaxError):
        return ''


# ----- # ----- # ----- Cleanup Data ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def clean_df(job, job_details, file_name):
    result = pd.merge(job, job_details, on="Id")
    result = result[['Id', 'Title', 'PostedDate', 'JobFamily', 'JobFunction', 'ShortDescriptionStr',
                     'PrimaryLocation', 'secondaryLocations', 'JobSchedule', 'ExternalDescriptionStr', 'CorporateDescriptionStr',
                     'OrganizationDescriptionStr', 'BusinessUnit', 'requisitionFlexFields']]

    # Clean columns
    result["PostedDate"] = pd.to_datetime(result["PostedDate"])
    result["PostedDate"] = result["PostedDate"].dt.date
    result["secondaryLocations"] = result["secondaryLocations"].apply(extract_locations)
    result["ExternalDescriptionStr"] = result["ExternalDescriptionStr"].apply(clean_html)
    result["CorporateDescriptionStr"] = result["CorporateDescriptionStr"].apply(clean_html)
    result["CorporateDescriptionStr"] = result['CorporateDescriptionStr'].replace("", "Not available.")
    result["OrganizationDescriptionStr"] = result["OrganizationDescriptionStr"].apply(clean_html)
    result["OrganizationDescriptionStr"] = result['OrganizationDescriptionStr'].replace("", "Not available.")
    result["requisitionFlexFields"] = result["requisitionFlexFields"].apply(extract_salary)

    # New columns
    result.insert(loc=0, column='Company', value='JPMC')
    result['Locations'] = result.apply(lambda x: f"{x['PrimaryLocation']}\n{x['secondaryLocations']}" if pd.notna(x['secondaryLocations']) and str(x['secondaryLocations']).strip() != "" else x['PrimaryLocation'], axis=1)
    result['Job Details'] = result['ExternalDescriptionStr'].astype(str)
    result['Additional Details'] = (
        "Job Category: " + result['JobFamily'].astype(str).str.cat([
        "Job Function: " + result['JobFunction'].astype(str),
        "Job Schedule: " + result['JobSchedule'].astype(str),
        "Business Unit: " + result['BusinessUnit'].astype(str),
        "Job Brief: " + result['ShortDescriptionStr'].astype(str)], sep="\n", na_rep="N/A"))
    result['Organization Details'] = (
        "Company Description: " + result['CorporateDescriptionStr'].fillna("Not available.") + "\n" +
        "Team Description: " + result['OrganizationDescriptionStr'].fillna("Not available."))
    result['URL'] = "https://jpmc.fa.oraclecloud.com/hcmUI/CandidateExperience/en/sites/CX_1001/job/" + result['Id'].astype(str)
    result['Full Information'] = (
        "Company: " + result['Company'].fillna("N/A").astype(str)).str.cat([
        "Title: " + result['Title'].fillna("N/A").astype(str),
        "Posting Date: " + result['PostedDate'].fillna("N/A").astype(str),
        "Locations: " + result['Locations'].fillna("N/A").astype(str),
        "Job Details: " + result['Job Details'].fillna("N/A").astype(str),
        "Job Category: " + result['JobFamily'].fillna("N/A").astype(str),
        "Job Function: " + result['JobFunction'].fillna("N/A").astype(str),
        "Job Schedule: " + result['JobSchedule'].fillna("N/A").astype(str),
        "Business Unit: " + result['BusinessUnit'].fillna("N/A").astype(str),
        "Job Brief: " + result['ShortDescriptionStr'].fillna("N/A").astype(str),
        # "Compensation: " + result['requisitionFlexFields'].fillna("N/A").astype(str),
        # "Company Description: " + result['CorporateDescriptionStr'].fillna("N/A").astype(str),
        # "Team Description: " + result['OrganizationDescriptionStr'].fillna("N/A").astype(str)
        ],sep="\n\n")
    result['Hash Key'] = (result["Company"].fillna("").astype(str) + "||" + result["Id"].fillna("").astype(str)).apply(lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest())

    # Drop Columns
    result.drop(columns=['JobFamily', 'JobFunction', 'ShortDescriptionStr', 'PrimaryLocation', 'secondaryLocations',
                        'JobSchedule', 'ExternalDescriptionStr', 'CorporateDescriptionStr', 'OrganizationDescriptionStr',
                         'BusinessUnit'], inplace=True)

    # Order and Rename
    result = result[['Hash Key', 'Company', 'Id', 'Title', 'PostedDate', 'Locations', 'Job Details', 'Additional Details', 'Organization Details', 'requisitionFlexFields', 'URL', 'Full Information']]
    result.columns = ['hash_key', 'company', 'job_id', 'title', 'posting_date', 'locations', 'job_details', 'additional_details', 'organization_details', 'compensation', 'url', 'full_information']
    result = result.astype(str)
    result.to_csv(file_name, header=True, index=False)
    return result

In [ ]:
# ----- # ----- # ----- Upload file to Databricks ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def upload_data(data_file):
  w = WorkspaceClient(
      host = userdata.get('DATABRICKS_HOST'),
      token = userdata.get('DATABRICKS_TOKEN')
  )

  volume_path = "/Volumes/workspace/jobs_raw/job_files/" + data_file.split('/')[-1]
  with open(data_file, "rb") as f:
      w.files.upload(volume_path, f)

  print(f"Successfully uploaded {data_file} to {volume_path}")

In [ ]:
# def run_pipeline_jpmc():
#   file_path = '/content/drive/MyDrive/Colab Notebooks/JobAI/jobs/JPMC.csv'
#   job_frame = fetch_jobs()
#   job_details_frame = fetch_job_details(job_frame)
#   final_df = clean_df(job_frame, job_details_frame, file_path)
#   upload_data(file_path)

# New Architecture

In [ ]:
job_frame = fetch_jobs()
upload_data('/content/drive/MyDrive/Colab Notebooks/JobAI/jobs/jpmc_jobs.parquet')
job_frame.reset_index(inplace=True, drop=False)

In [ ]:
job_details_frame = fetch_job_details(job_frame)
upload_data('/content/drive/MyDrive/Colab Notebooks/JobAI/jobs/jpmc_jobs_details.parquet')